# WSJ 2027 - Gruppindelning

Cluster participants into groups of 36 based on rules.

**Phase 1:** Deltagare med rundresa  
**Phase 2:** Other categories (direktresa, IST, etc.)

## Rules
1. Each participant gets their friend wish respected (Medlemsnummer person 1 & 2)
2. *(More rules to be defined)*

## Group size: 36 persons

In [1]:
# Cell 1: Imports and API configuration
import requests
import json
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from datetime import date

SCOUTNET_API_ID = "43490"
SCOUTNET_API_KEY = "REDACTED"
SCOUTNET_URL = "https://www.scoutnet.se/api/project/get/participants"

# Questions form API (for discovering question IDs)
QUESTIONS_API_KEY = "REDACTED"
QUESTIONS_URL = "https://www.scoutnet.se/api/project/get/questions"

WSJ_START = date(2027, 7, 29)
GROUP_SIZE = 36

# Fee categories
DELTAGARE_FEES = {'27561', '25694'}  # direktresa, rundresa
DELTAGARE_RUNDRESA = '25694'
DELTAGARE_DIREKTRESA = '27561'
IST_FEES = {'25702', '25696', '25697', '25693'}

# Question IDs (from Scoutnet form 39188)
Q_FRIEND_1_MEMBER_NO = '87660'  # "Medlemsnummer person 1"
Q_FRIEND_1_NAME = '87662'       # "Person 1" (free text name)
Q_FRIEND_2_MEMBER_NO = '87663'  # "Medlemsnummer person 2"
Q_FRIEND_2_NAME = '87665'       # "Person 2" (free text name)

print("Configuration loaded")
print(f"Group size: {GROUP_SIZE}")

Configuration loaded
Group size: 36


In [2]:
# Cell 2: Fetch and parse participants
response = requests.get(SCOUTNET_URL, auth=(SCOUTNET_API_ID, SCOUTNET_API_KEY))
response.raise_for_status()
raw_data = response.json()
participants_raw = raw_data.get('participants', {})
print(f"Fetched {len(participants_raw)} participants")

Fetched 1145 participants


In [3]:
# Cell 3: Build participant DataFrame with age validation

def exact_age(birth_date, ref_date):
    """Calculate exact age in whole years at ref_date."""
    years = ref_date.year - birth_date.year
    if (ref_date.month, ref_date.day) < (birth_date.month, birth_date.day):
        years -= 1
    return years

def get_question(questions, qid):
    """Safely get a question answer, handling list/dict."""
    if not isinstance(questions, dict):
        return ''
    val = questions.get(qid, '')
    if val is None:
        return ''
    return str(val).strip()

rows = []
skipped = []

for mid, p in participants_raw.items():
    if p.get('cancelled'):
        continue
    
    fee_id = str(p.get('fee_id', ''))
    dob = p.get('date_of_birth', '')
    name = f"{p.get('first_name', '')} {p.get('last_name', '')}"
    
    birth = None
    if dob:
        try:
            birth = date.fromisoformat(dob)
        except ValueError:
            pass
    
    # Age validation
    if birth is None:
        skipped.append(f"  NO DOB: {name} fee={fee_id}")
        continue
    
    age = exact_age(birth, WSJ_START)
    
    if fee_id in DELTAGARE_FEES and (age < 14 or age >= 18):
        skipped.append(f"  DELTAGARE wrong age: {name} born {dob} (age {age})")
        continue
    if fee_id in IST_FEES and age < 18:
        skipped.append(f"  IST too young: {name} born {dob} (age {age})")
        continue
    
    # Determine category
    if fee_id in DELTAGARE_FEES:
        category = 'deltagare'
    elif fee_id in IST_FEES:
        category = 'ist'
    else:
        category = 'deltagare' if 14 <= age <= 17 else 'ist'
    
    # Extract kår info
    membership = p.get('primary_membership_info', {})
    if isinstance(membership, dict):
        kar = membership.get('group_name', '')
        district = membership.get('district_name', '')
        region = membership.get('region_name', '')
    else:
        kar, district, region = '', '', ''
    
    # Extract friend wishes from questions
    questions = p.get('questions', {})
    friend_1 = get_question(questions, Q_FRIEND_1_MEMBER_NO)
    friend_2 = get_question(questions, Q_FRIEND_2_MEMBER_NO)
    friend_1_name = get_question(questions, Q_FRIEND_1_NAME)
    friend_2_name = get_question(questions, Q_FRIEND_2_NAME)
    
    # Clean friend member numbers (ignore '0' and empty)
    friend_1 = friend_1 if friend_1 and friend_1 != '0' else ''
    friend_2 = friend_2 if friend_2 and friend_2 != '0' else ''
    
    rows.append({
        'member_no': str(p.get('member_no', mid)),
        'name': name,
        'birth_date': dob,
        'age': age,
        'sex': p.get('sex', 0),
        'fee_id': fee_id,
        'category': category,
        'travel': 'rundresa' if fee_id == DELTAGARE_RUNDRESA else ('direktresa' if fee_id == DELTAGARE_DIREKTRESA else 'other'),
        'kar': kar,
        'district': district,
        'region': region,
        'friend_1': friend_1,
        'friend_2': friend_2,
        'friend_1_name': friend_1_name,
        'friend_2_name': friend_2_name,
        'group': None,  # To be assigned
    })

df = pd.DataFrame(rows)

print(f"Total active participants: {len(df)}")
print(f"Skipped: {len(skipped)}")
print(f"\nBy category:")
print(df['category'].value_counts().to_string())
print(f"\nBy travel type:")
print(df['travel'].value_counts().to_string())
print(f"\nFriend wishes:")
print(f"  With friend 1 member no: {(df['friend_1'] != '').sum()}")
print(f"  With friend 2 member no: {(df['friend_2'] != '').sum()}")
print(f"  With friend 1 name (text): {(df['friend_1_name'] != '').sum()}")
print(f"  With friend 2 name (text): {(df['friend_2_name'] != '').sum()}")

if skipped:
    print(f"\nSkipped participants:")
    for s in skipped:
        print(s)

Total active participants: 1113
Skipped: 3

By category:
category
deltagare    931
ist          182

By travel type:
travel
rundresa      752
other         182
direktresa    179

Friend wishes:
  With friend 1 member no: 603
  With friend 2 member no: 339
  With friend 1 name (text): 55
  With friend 2 name (text): 38

Skipped participants:
  DELTAGARE wrong age: Juniper Grapatin born 2009-07-07 (age 18)
  IST too young: Daniel Kursitis Kuzyaev born 2009-12-04 (age 17)
  DELTAGARE wrong age: Emma Tonell born 1980-02-20 (age 47)


In [4]:
# Cell 4: Filter to Phase 1 - Deltagare rundresa only

df_rundresa = df[df['travel'] == 'rundresa'].copy().reset_index(drop=True)

n_groups = len(df_rundresa) // GROUP_SIZE
remainder = len(df_rundresa) % GROUP_SIZE

print(f"=== Phase 1: Deltagare rundresa ===")
print(f"Participants: {len(df_rundresa)}")
print(f"Groups of {GROUP_SIZE}: {n_groups} full groups")
print(f"Remainder: {remainder} (will need to distribute across groups)")
print(f"\nBy region:")
print(df_rundresa['region'].value_counts().to_string())
print(f"\nBy age:")
print(df_rundresa['age'].value_counts().sort_index().to_string())
print(f"\nBy sex:")
sex_map = {'0': 'Okänt', '1': 'Man', '2': 'Kvinna', '3': 'Annat', '4': 'Icke-binär',
           0: 'Okänt', 1: 'Man', 2: 'Kvinna', 3: 'Annat', 4: 'Icke-binär'}
print(df_rundresa['sex'].map(sex_map).value_counts().to_string())

=== Phase 1: Deltagare rundresa ===
Participants: 752
Groups of 36: 20 full groups
Remainder: 32 (will need to distribute across groups)

By region:
region
Region Stockholm    228
Region Södra        186
Region Västra       150
Region Norr/Mitt     74
Region Östra         46
                     21

By age:
age
14    183
15    225
16    177
17    167

By sex:
sex
Kvinna    384
Man       364
Annat       4


In [ ]:
# Cell 5: Friend wish analysis (runs before text matching)
# Build friend graph: member_no -> [friend_1, friend_2]

member_set = set(df_rundresa['member_no'].values)
all_members = set(df['member_no'].values)

# Count text-only wishes before resolving
text_only_count = sum(1 for _, r in df_rundresa.iterrows()
                      if (not r['friend_1'] and r['friend_1_name']) or
                         (not r['friend_2'] and r['friend_2_name']))

# Summary before text matching
print(f"=== Friend Wishes (before text matching) ===")
print(f"With friend 1 (member no): {(df_rundresa['friend_1'] != '').sum()}")
print(f"With friend 2 (member no): {(df_rundresa['friend_2'] != '').sum()}")
print(f"With text name only (no member no): {text_only_count}")
print(f"Without any friend wish: {((df_rundresa['friend_1'] == '') & (df_rundresa['friend_2'] == '') & (df_rundresa['friend_1_name'] == '') & (df_rundresa['friend_2_name'] == '')).sum()}")

In [ ]:
# Cell 5b: Resolve text-only friend wishes via name matching
import re
from difflib import SequenceMatcher

def normalize_name(name):
    return re.sub(r'\s+', ' ', name.strip().lower())

def parse_friend_text(text):
    """Parse free-text friend wish into (name_part, kar_hint)."""
    text = text.strip()
    # Skip generic wishes (not a specific person)
    skip_patterns = ['gärna någon', 'övriga deltagare', 'scouter från']
    if any(p in text.lower() for p in skip_patterns):
        return None, text
    
    kar_hint = ''
    name_part = text
    
    if ',' in text:
        parts = [p.strip() for p in text.split(',')]
        name_part = parts[0]
        kar_hint = ' '.join(parts[1:])
    elif ' - ' in text:
        parts = [p.strip() for p in text.split(' - ', 1)]
        name_part = parts[0]
        kar_hint = parts[1]
    else:
        kar_starters = ['scoutkår', 'sjöscoutkår', 'scouterkår', 'scoutskår',
                        'equmenia', 'kåren', 'scoutkåren']
        text_lower = text.lower()
        for kw in kar_starters:
            idx = text_lower.find(kw)
            if idx > 0:
                before = text[:idx].rstrip()
                bwords = before.split()
                if len(bwords) > 2:
                    name_part = ' '.join(bwords[:2])
                    kar_hint = ' '.join(bwords[2:]) + ' ' + text[idx:]
                    kar_hint = kar_hint.strip()
                break
        else:
            words = text.split()
            if len(words) > 3:
                name_part = ' '.join(words[:2])
                kar_hint = ' '.join(words[2:])
    
    return name_part.strip(), kar_hint.strip()

def fuzzy_match_name(query_name, name_lookup, kar_hint='', threshold=0.75):
    """Find best matching participant by name with optional kår boost."""
    query_norm = normalize_name(query_name)
    
    # 1. Exact match
    if query_norm in name_lookup:
        matches = name_lookup[query_norm]
        if len(matches) == 1:
            return matches[0], 'exact', 1.0
        if kar_hint:
            kar_lower = kar_hint.lower()
            for m in matches:
                if kar_lower in m['kar'].lower() or m['kar'].lower() in kar_lower:
                    return m, 'exact+kår', 1.0
        return matches[0], 'exact(ambiguous)', 1.0
    
    # 2. Try first + last word
    query_words = query_norm.split()
    if len(query_words) >= 2:
        short_query = query_words[0] + ' ' + query_words[-1]
        if short_query in name_lookup:
            matches = name_lookup[short_query]
            if len(matches) == 1:
                return matches[0], 'first+last', 0.95
    
    # 3. Fuzzy matching
    best_score = 0
    best_match = None
    for norm_name, candidates in name_lookup.items():
        score = SequenceMatcher(None, query_norm, norm_name).ratio()
        cand_words = norm_name.split()
        if len(cand_words) >= 2:
            short_cand = cand_words[0] + ' ' + cand_words[-1]
            score = max(score, SequenceMatcher(None, query_norm, short_cand).ratio())
        if len(query_words) >= 2:
            short_q = query_words[0] + ' ' + query_words[-1]
            score = max(score, SequenceMatcher(None, short_q, norm_name).ratio())
        if score > best_score:
            best_score = score
            best_match = candidates[0]
    
    if best_score >= threshold and best_match:
        method = f'fuzzy({best_score:.2f})'
        if kar_hint:
            kar_lower = kar_hint.lower()
            if kar_lower in best_match['kar'].lower() or best_match['kar'].lower() in kar_lower:
                method += '+kår'
                best_score = min(best_score + 0.1, 1.0)
        return best_match, method, best_score
    
    return None, 'no_match', best_score

# Build name lookup
name_lookup = defaultdict(list)
for _, row in df.iterrows():
    norm = normalize_name(row['name'])
    name_lookup[norm].append({
        'member_no': row['member_no'], 'name': row['name'],
        'kar': row['kar'], 'travel': row['travel'],
    })

# Collect text-only wishes
text_wishes = []
for _, row in df_rundresa.iterrows():
    if not row['friend_1'] and row['friend_1_name']:
        text_wishes.append({
            'wisher_member_no': row['member_no'], 'wisher': row['name'],
            'friend_text': row['friend_1_name'], 'slot': 'friend_1',
        })
    if not row['friend_2'] and row['friend_2_name']:
        text_wishes.append({
            'wisher_member_no': row['member_no'], 'wisher': row['name'],
            'friend_text': row['friend_2_name'], 'slot': 'friend_2',
        })

# Match and validate
verified = []
unresolved = []
skipped_generic = []

for tw in text_wishes:
    name_part, kar_hint = parse_friend_text(tw['friend_text'])
    if name_part is None:
        skipped_generic.append(tw)
        continue
    
    match, method, score = fuzzy_match_name(name_part, name_lookup, kar_hint)
    
    if match:
        # Validate: first or last name must match
        parsed_words = normalize_name(name_part).split()
        match_words = normalize_name(match['name']).split()
        if score < 0.90:
            p_first, p_last = parsed_words[0], parsed_words[-1] if len(parsed_words) > 1 else ''
            m_first, m_last = match_words[0], match_words[-1] if len(match_words) > 1 else ''
            if p_first != m_first and p_last != m_last:
                unresolved.append({**tw, 'reason': f'name mismatch ({match["name"]})'})
                continue
        verified.append({**tw, 'match': match, 'method': method, 'score': score})
    else:
        unresolved.append({**tw, 'reason': 'no match found'})

# Apply matches to DataFrame
updates = 0
for v in verified:
    idx = df_rundresa[df_rundresa['member_no'] == v['wisher_member_no']].index
    if len(idx) > 0 and df_rundresa.at[idx[0], v['slot']] == '':
        df_rundresa.at[idx[0], v['slot']] = v['match']['member_no']
        # Also update main df
        main_idx = df[df['member_no'] == v['wisher_member_no']].index
        if len(main_idx) > 0:
            df.at[main_idx[0], v['slot']] = v['match']['member_no']
        updates += 1

print(f"=== Text Friend Name Matching ===")
print(f"Text-only wishes: {len(text_wishes)}")
print(f"Matched & applied: {len(verified)}")
print(f"Generic wishes (not a person): {len(skipped_generic)}")
print(f"Unresolved (friend not in project): {len(unresolved)}")

print(f"\nMatched:")
for v in sorted(verified, key=lambda x: x['wisher']):
    m = v['match']
    print(f"  {v['wisher']} -> {m['name']} ({m['kar']}) [{m['travel']}] via {v['method']}")

if unresolved:
    print(f"\nUnresolved:")
    for u in sorted(unresolved, key=lambda x: x['wisher']):
        print(f"  {u['wisher']} -> \"{u['friend_text']}\" ({u['reason']})")

if skipped_generic:
    print(f"\nGeneric wishes (skipped):")
    for s in skipped_generic:
        print(f"  {s['wisher']} -> \"{s['friend_text']}\"")

In [ ]:
# Cell 6: Final friend wish summary (after text matching)

member_set = set(df_rundresa['member_no'].values)
all_members = set(df['member_no'].values)

friend_wishes = {}
for _, row in df_rundresa.iterrows():
    wishes = []
    if row['friend_1'] and row['friend_1'] in member_set:
        wishes.append(row['friend_1'])
    if row['friend_2'] and row['friend_2'] in member_set:
        wishes.append(row['friend_2'])
    if wishes:
        friend_wishes[row['member_no']] = wishes

# Mutual pairs
mutual_pairs = set()
one_way = []
for member, wishes in friend_wishes.items():
    for friend in wishes:
        if friend in friend_wishes and member in friend_wishes[friend]:
            pair = tuple(sorted([member, friend]))
            mutual_pairs.add(pair)
        else:
            one_way.append((member, friend))

# Cross-category and unknown
cross_category = []
unknown_member = []
for _, row in df_rundresa.iterrows():
    for f_col in ['friend_1', 'friend_2']:
        fid = row[f_col]
        if fid and fid not in member_set:
            if fid in all_members:
                friend_row = df[df['member_no'] == fid].iloc[0]
                cross_category.append((row['name'], friend_row['name'], friend_row['travel']))
            else:
                unknown_member.append((row['name'], fid))

print(f"=== Final Friend Wish Summary (after text matching) ===")
print(f"Participants with >=1 wish in rundresa: {len(friend_wishes)}")
print(f"Mutual pairs (both wish each other): {len(mutual_pairs)}")
print(f"One-way wishes: {len(one_way)}")
print(f"Without any resolved friend wish: {len(df_rundresa) - len(friend_wishes)}")
print(f"Cross-category wishes: {len(cross_category)}")
print(f"Unknown member numbers: {len(unknown_member)}")

if cross_category:
    print(f"\nCross-category ({len(cross_category)}):")
    for name, fname, ftravel in cross_category[:10]:
        print(f"  {name} -> {fname} ({ftravel})")
    if len(cross_category) > 10:
        print(f"  ... and {len(cross_category)-10} more")